# 🌲 Section 9 — Arbres de Décision : Classifier les Équipements Énergétiques (Sèche-Linge, Cuit-Vapeur, Recharge VE)

Bienvenue dans ce neuvième module du parcours **Data Science & Python** !

Dans les modules précédents :
- **Sections 1 & 2** : Fondations du calcul matriciel avec **NumPy**.
- **Sections 3 & 4** : Premiers modèles d'apprentissage (**Perceptron** et **Descente de Gradient**).
- **Sections 5 & 6** : Manipulation des séries temporelles avec **Pandas** et analyse des corrélations.
- **Sections 7 & 8** : Extraction de signatures fréquentielles et temporelles sur les signaux domotiques (**Energy Lab**).

Jusqu'à présent, les modèles linéaires comme le Perceptron traçaient une frontière de décision continue et oblique dans l'espace des données. Cependant, dans le monde industriel et domotique, les ingénieurs et auditeurs énergétiques exigent souvent des **modèles explicables à 100%**, capables d'énoncer des règles logiques claires en langage naturel :
> *« SI la puissance dépasse 3 000 W ET que le signal est continu sans coupure, ALORS c'est une recharge de véhicule électrique. »*

C'est exactement la force des **Arbres de Décision** (*Decision Trees*), l'un des algorithmes les plus intuitifs, polyvalents et puissants du Machine Learning !

---

### 🏠 Le Cas d'Étude : Distinguer les gros consommateurs du foyer
Dans une maison connectée, plusieurs appareils appellent de très fortes puissances, mais avec des signatures bien distinctes :
1. **Le Cuit-Vapeur** : Puissance modérée (~1 100 W), cycles courts (25-45 min), régulation par thermostat.
2. **Le Sèche-Linge** : Puissance élevée (~2 450 W), durée moyenne (50-90 min), signal très haché (forte variance due aux bascules du thermostat et à l'inversion du tambour).
3. **La Recharge de Véhicule Électrique (VE)** : Puissance massive et continue (~3 700 W sur prise 16A ou 7 400 W sur borne), durée de plusieurs heures, profil ultra-plat sans aucune oscillation.

Comment un arbre de décision apprend-il à poser les **bonnes questions au bon moment** pour séparer ces appareils sans jamais se tromper ?

---

### 🗺️ Votre parcours d'apprentissage dans cette section :
0. **👁️ Coup d'œil visuel initial** — Signatures temporelles et plan de séparation 2D
1. **Section 1** — Pourquoi les Arbres de Décision ? De l'hyperplan aux frontières orthogonales
2. **Section 2** — Comment l'arbre choisit-il ses questions ? L'Impureté de Gini et l'Entropie
3. **Section 3** — Génération et Préparation du Benchmark Multi-Appareils
4. **Section 4** — Entraînement et Visualisation de l'Arbre avec Scikit-Learn (`plot_tree`)
5. **Section 5** — Évaluation des Performances et Matrice de Confusion (Comparaison avec Dummy et KNN)
6. **Section 6** — Le Risque de Surapprentissage (*Overfitting*) et les Hyperparamètres (`max_depth`)
7. **Section 7** — 🏆 Défi Final : La Classification d'une Session Mystère
8. **Section 8** — Synthèse & Tremplin vers la Section 10 (Random Forest)


In [ ]:
# Programme 1 — Initialisation & Outils interactifs (QCM et validation)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random as _random_qcm
import ipywidgets as w
from IPython.display import display

# Outils Scikit-Learn indispensables
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score

# Configuration graphique
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (11, 4.5)
plt.rcParams['font.size'] = 10

# Relances humoristiques adaptées aux arbres de décision
_RELANCES_HUMOUR = [
    "Gini te dit que ce nœud est encore bien trop impur !",
    "Attention au surapprentissage (overfitting) : ton arbre a mémorisé le bruit par cœur !",
    "As-tu vérifié max_depth ? Un arbre trop profond n'est plus généralisable !",
    "Une matrice de confusion se lit : Réel en ligne, Prédit en colonne !",
    "Attention aux fuites de données (Data Leakage) : entraîne toujours sur le train set !",
    "Même Leo Breiman a dû élaguer ses premiers arbres de décision !"
]

def qcm(question, options, correct, explication=""):
    """
    Affiche un QCM interactif avec ipywidgets.
    correct : index (int) ou liste d'index de la/des bonne(s) réponse(s).
    """
    if isinstance(correct, int):
        correct = [correct]

    boutons = w.RadioButtons(options=options, layout=w.Layout(width='100%'))
    sortie  = w.Output()
    titre   = w.HTML(f"<b>❓ {question}</b>")

    def verifier(change):
        idx = options.index(change['new'])
        sortie.clear_output()
        with sortie:
            if idx in correct:
                display(w.HTML(
                    f"<div style='padding:10px;border-radius:8px;background:#d4edda;color:#155724;border-left:5px solid #28a745'>"
                    f"✅ <b>Bravo, c'est exact !</b><br>{explication}</div>"
                ))
            else:
                humour = _random_qcm.choice(_RELANCES_HUMOUR)
                display(w.HTML(
                    f"<div style='padding:10px;border-radius:8px;background:#f8d7da;color:#721c24;border-left:5px solid #dc3545'>"
                    f"❌ <b>Pas tout à fait...</b> {humour}<br><br>"
                    f"<b>Explication :</b> {explication}</div>"
                ))

    boutons.observe(verifier, names='value')
    display(w.VBox([titre, boutons, sortie]))

def check(nom, valeur, attendu, tol=1e-3):
    """
    Vérifie la valeur d'une variable ou métrique dans un exercice.
    Affiche un retour visuel vert ou rouge.
    """
    ok = False
    try:
        if isinstance(attendu, (pd.Series, pd.DataFrame)) and isinstance(valeur, (pd.Series, pd.DataFrame)):
            if attendu.shape == valeur.shape:
                if isinstance(attendu, pd.Series):
                    ok = bool(np.allclose(valeur.dropna().values, attendu.dropna().values, atol=tol))
                else:
                    ok = list(valeur.columns) == list(attendu.columns)
        elif isinstance(attendu, np.ndarray) and isinstance(valeur, np.ndarray):
            ok = valeur.shape == attendu.shape and bool(np.allclose(valeur, attendu, atol=tol))
        elif isinstance(attendu, (int, float, np.integer, np.floating)) and isinstance(valeur, (int, float, np.integer, np.floating)):
            ok = abs(float(valeur) - float(attendu)) <= tol
        elif isinstance(attendu, (tuple, list)) and isinstance(valeur, (tuple, list)):
            ok = len(valeur) == len(attendu) and all(
                (abs(float(v) - float(a)) <= tol if isinstance(a, (int, float)) else v == a)
                for v, a in zip(valeur, attendu)
            )
        else:
            ok = bool(valeur == attendu)
    except Exception:
        ok = False

    if ok:
        display(w.HTML(
            f"<div style='padding:8px 12px;border-radius:6px;background:#d4edda;color:#155724;border-left:4px solid #28a745'>"
            f"✅ <b>{nom}</b> est correct ! (Valeur : <code>{repr(valeur)}</code>)</div>"
        ))
    else:
        display(w.HTML(
            f"<div style='padding:8px 12px;border-radius:6px;background:#f8d7da;color:#721c24;border-left:4px solid #dc3545'>"
            f"❌ <b>{nom}</b> n'est pas encore la valeur attendue.<br>"
            f"Ta valeur : <code>{repr(valeur)}</code><br>"
            f"Valeur attendue : <code>{repr(attendu)}</code></div>"
        ))

print(f"✅ Environnement Section 9 initialisé avec Scikit-Learn et Matplotlib.")
print("✅ Fonctions qcm(), check() et DecisionTreeClassifier prêtes à l'emploi.")


---
## 👁️ Coup d'œil visuel initial : Les signatures physiques de nos 3 appareils

Avant d'ajuster le moindre modèle, **regardons la forme brute du signal électrique** émis par un cuit-vapeur, un sèche-linge et une recharge de voiture électrique, puis projetons leurs caractéristiques (*features*) dans un plan 2D.

Exécutez le **Programme 2** ci-dessous :


In [ ]:
# Programme 2 — Signatures temporelles et plan de séparation 2D
rng_demo = np.random.default_rng(42)
t_demo = np.arange(0, 3600, 10) # 1 heure de relevés à 10s d'intervalle

# 1. Profil Cuit-vapeur (pulses carrés thermostat à 1 100 W)
sig_cuit = np.where((t_demo % 400) < 280, 1100 + rng_demo.normal(0, 20, len(t_demo)), 30)
sig_cuit[180:] = 30 # s'arrête au bout de 30 min

# 2. Profil Sèche-linge (grosses bascules 2 400 W thermostat + rotation tambour)
sig_seche = np.where((t_demo % 350) < 220, 2450 + rng_demo.normal(0, 60, len(t_demo)), 180 + rng_demo.normal(0, 20, len(t_demo)))

# 3. Profil Recharge VE (puissance massive et plate à 3 700 W)
sig_ve = np.full(len(t_demo), 3700.0) + rng_demo.normal(0, 15, len(t_demo))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# 1. Tracé temporel
ax1.plot(t_demo / 60, sig_ve, color='#2a9d8f', linewidth=2, label="Recharge VE (3 700 W plat)")
ax1.plot(t_demo / 60, sig_seche, color='#e76f51', linewidth=1.5, label="Sèche-linge (2 450 W haché)")
ax1.plot(t_demo / 60, sig_cuit, color='#457b9d', linewidth=1.5, label="Cuit-vapeur (1 100 W court)")
ax1.set_title("1. Forme d'onde brute des 3 appareils sur 1 heure", fontweight='bold')
ax1.set_xlabel("Temps (minutes)")
ax1.set_ylabel("Puissance (Watts)")
ax1.legend(loc='center right')

# 2. Espace des Features (Puissance Max vs Écart-Type)
# Génération de quelques points pour illustrer le nuage 2D
n_pts = 30
ax2.scatter(rng_demo.normal(1100, 50, n_pts), rng_demo.normal(320, 25, n_pts), color='#457b9d', s=45, label="Cuit-vapeur")
ax2.scatter(rng_demo.normal(2450, 80, n_pts), rng_demo.normal(720, 40, n_pts), color='#e76f51', s=45, label="Sèche-linge")
ax2.scatter(rng_demo.normal(3700, 30, n_pts), rng_demo.normal(45, 10, n_pts), color='#2a9d8f', s=45, label="Recharge VE")

# Lignes de coupe orthogonales (style arbre de décision)
ax2.axvline(1800, color='black', linestyle='--', linewidth=1.8, label="Split 1 : P_max = 1 800 W")
ax2.axvline(3100, color='darkred', linestyle='--', linewidth=1.8, label="Split 2 : P_max = 3 100 W")

ax2.set_title("2. Séparation orthogonale dans l'espace des features", fontweight='bold')
ax2.set_xlabel("Puissance maximale (Watts)")
ax2.set_ylabel("Écart-type de la puissance (Watts)")
ax2.legend()

plt.tight_layout()
plt.show()


### 🔍 Ce que révèle ce graphique d'accroche :
1. **Dans le domaine temporel (à gauche)** :
   - Le **cuit-vapeur** est facilement identifiable par son niveau de puissance modéré (~1 100 W).
   - Le **sèche-linge** et la **recharge VE** sont tous deux très puissants (> 2 000 W), mais leur dynamique est opposée : le sèche-linge oscille continuellement ($\sigma \approx 700\text{ W}$), tandis que la voiture charge à puissance rigoureusement constante ($\sigma \approx 45\text{ W}$).
2. **Dans l'espace des caractéristiques (à droite)** :
   - Les trois classes forment des **paquets nettement séparés**.
   - Mieux encore : pour les séparer, nous n'avons pas besoin d'une équation complexe ou oblique ! Deux simples lignes verticales (ex: *« Est-ce que $P_{\max} < 1800\text{ W}$ ? »* puis *« Est-ce que $P_{\max} > 3100\text{ W}$ ? »*) suffisent à isoler chaque groupe.

C'est exactement la géométrie élémentaire d'un **Arbre de Décision** !


---
## 1. Pourquoi les Arbres de Décision ? De l'hyperplan aux frontières orthogonales

### 📐 Géométrie comparée : Perceptron vs Arbre de décision
Dans les **Sections 3 et 4**, nous avons étudié le Perceptron et la Régression Logistique. Ces modèles tracent un **hyperplan continu oblique** d'équation :
$$w_1 x_1 + w_2 x_2 + b = 0$$

Un **Arbre de Décision**, en revanche, procède par dichotomies successives en posant une question sur **une seule variable à la fois** :
$$\text{Variable}_j \le \text{Seuil} \quad ?$$

Géométriquement, chaque question divise l'espace par une droite **strictement perpendiculaire à l'axe de la variable**. En combinant plusieurs questions, l'arbre découpe l'espace en **boîtes ou rectangles orthogonaux** (*hyper-rectangles*).

```
   Perceptron (Ligne oblique)              Arbre de Décision (Découpage en boîtes)
        x2 ^                                    x2 ^
           |     /                                 |      |   Zone B
           |    /   Zone B                         |------+-------
           |   /                                   |Zone A|   Zone C
           +----------> x1                         +----------> x1
```

### 🌟 Les atouts majeurs de l'arbre en industrie :
1. **Interprétabilité totale (Modèle « boîte blanche »)** : Contrairement à un réseau de neurones profond, n'importe quel ingénieur ou technicien peut inspecter le cheminement logique.
2. **Aucune normalisation requise** : L'arbre ne fait que des comparaisons de type $\le$. Que la puissance soit en Watts ($[0, 5000]$) ou que le duty cycle soit entre $0$ et $1$, l'arbre ne nécessite ni Min-Max ni Z-score !
3. **Sélection naturelle des variables** : Les variables inutiles sont tout simplement ignorées lors du fractionnement.


In [ ]:
# Programme 3 — QCM 1
qcm(
    "Pourquoi dit-on que les frontières de décision d'un arbre de décision standard sont 'orthogonales aux axes' ?",
    [
        "Parce que l'arbre ne fonctionne qu'avec des angles de 45 degrés.",
        "Parce qu'à chaque nœud, l'arbre compare une seule variable à un seuil fixe (ex: X_j <= seuil), ce qui produit une coupure parallèle aux autres axes.",
        "Parce que les arbres de décision utilisent obligatoirement le théorème de Pythagore.",
        "Parce que l'arbre ne peut traiter que des nombres négatifs."
    ],
    correct=1,
    explication="Chaque nœud de décision évalue une condition sur une seule caractéristique (feature), sans la combiner avec les autres. Dans un espace 2D (x1, x2), tester 'x1 <= seuil' crée une ligne verticale parfaite, et 'x2 <= seuil' une ligne horizontale parfaite."
)


---
## 2. Comment l'arbre choisit-il ses questions ? L'Impureté de Gini et l'Entropie

À chaque étape, l'algorithme doit choisir :
1. **Quelle variable tester ?** (ex: `puissance_max_W` ou `ecart_type_W` ?)
2. **Quel seuil choisir ?** (ex: $1800\text{ W}$ ou $2200\text{ W}$ ?)

Pour trouver le « meilleur » découpage, l'arbre mesure la **pureté** des sous-groupes créés.

---

### 📉 L'Indice d'Impureté de Gini
Soit un nœud contenant des exemples répartis entre $K$ classes avec les proportions $p_1, p_2, \dots, p_K$ (où $\sum p_k = 1$).
L'**Impureté de Gini** est définie par :
$$G = 1 - \sum_{k=1}^K p_k^2$$

- **Nœud pur ($G = 0$)** : Tous les exemples appartiennent à la même classe (ex: 100% cuit-vapeur $\rightarrow p_1 = 1$, $G = 1 - 1^2 = 0$).
- **Nœud impur maximal** : Pour 2 classes équilibrées (50% / 50%) :
  $$G = 1 - (0.5^2 + 0.5^2) = 1 - 0.5 = 0.5$$

### 🧪 Le Gain d'Information (Réduction de Gini) :
L'arbre teste tous les seuils possibles et retient celui qui maximise la baisse d'impureté pondérée :
$$\Delta G = G_{\text{parent}} - \left(\frac{N_{\text{gauche}}}{N} G_{\text{gauche}} + \frac{N_{\text{droite}}}{N} G_{\text{droite}}\right)$$

---

### ✏️ Exercice 1 — Calcul manuel de l'impureté de Gini

Imaginons un nœud contenant **40 observations d'appareils** :
- **30** sessions de cuit-vapeur
- **10** sessions de sèche-linge

1. Calculez les proportions $p_{\text{cuit}} = 30 / 40$ et $p_{\text{seche}} = 10 / 40$.
2. Calculez l'impureté de Gini $G = 1 - (p_{\text{cuit}}^2 + p_{\text{seche}}^2)$ et stockez-la dans `gini_exercice`.

<details>
<summary>💡 Cliquez ici pour afficher un indice ou la solution</summary>

```python
p_cuit = 30 / 40
p_seche = 10 / 40
gini_exercice = float(1.0 - (p_cuit**2 + p_seche**2))
```
</details>


In [ ]:
# Programme 4 — Démonstration : Calcul de Gini sur différents mélanges
def calculer_gini(proportions):
    return 1.0 - sum(p**2 for p in proportions)

print("Exemples d'impureté de Gini (2 classes) :")
print(f"  • Nœud 100% pur (1.0, 0.0)      : Gini = {calculer_gini([1.0, 0.0]):.3f}")
print(f"  • Nœud majoritaire (0.9, 0.1)    : Gini = {calculer_gini([0.9, 0.1]):.3f}")
print(f"  • Nœud mélangé (0.75, 0.25)     : Gini = {calculer_gini([0.75, 0.25]):.3f}")
print(f"  • Nœud d'impureté max (0.5, 0.5) : Gini = {calculer_gini([0.5, 0.5]):.3f}")


In [ ]:
# Programme 5 — Exercice 1
# À toi de jouer : calcule l'impureté de Gini pour 30 cuit-vapeurs et 10 sèche-linges :

p_cuit = ...
p_seche = ...
gini_exercice = ...


In [ ]:
# Programme 6 — Test Exercice 1
# Exécutez cette cellule pour valider vos réponses
check("gini_exercice", gini_exercice, 0.375, tol=1e-4)


---
## 3. Génération et Préparation du Benchmark Multi-Appareils

Pour entraîner notre classifieur, nous simulons un jeu de données réaliste de **150 sessions de fonctionnement** (50 par appareil) caractérisées par 4 features extraites du signal :
- `puissance_max_W` : Puissance de crête observée lors de la session.
- `ecart_type_W` : Variabilité de la puissance (faible pour la recharge VE, très forte pour le sèche-linge).
- `duree_min` : Durée totale de la session en minutes.
- `duty_cycle` : Taux d'activité effectif ($0.55$ à $1.0$).

---

### ✏️ Exercice 2 — Séparation Train / Test

Dans la cellule ci-dessous, le DataFrame `df_appareils` est généré avec les 150 sessions.
Séparez les données en un jeu d'entraînement (**70%**) et un jeu de test (**30%**) à l'aide de `train_test_split` :
- `X = df_appareils[['puissance_max_W', 'ecart_type_W', 'duree_min', 'duty_cycle']]`
- `y = df_appareils['appareil']`
- Utilisez `test_size=0.3`, `random_state=42` et `stratify=y` (pour garantir que chaque ensemble contienne exactement 33% de chaque classe).

<details>
<summary>💡 Cliquez ici pour afficher un indice ou la solution</summary>

```python
X = df_appareils[['puissance_max_W', 'ecart_type_W', 'duree_min', 'duty_cycle']]
y = df_appareils['appareil']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
```
</details>


In [ ]:
# Programme 7 — Génération du dataset de 150 sessions
rng_sim = np.random.default_rng(42)
n_samples = 50

# 1. Cuit-vapeur : ~1100 W, court, variance modérée
cuit_p_max = rng_sim.normal(1100, 60, n_samples)
cuit_duree = rng_sim.uniform(25, 45, n_samples)
cuit_std   = rng_sim.normal(320, 25, n_samples)
cuit_duty  = rng_sim.uniform(0.65, 0.75, n_samples)

# 2. Sèche-linge : ~2450 W, moyen, très forte variance (thermostat pulsé)
seche_p_max = rng_sim.normal(2450, 90, n_samples)
seche_duree = rng_sim.uniform(50, 90, n_samples)
seche_std   = rng_sim.normal(720, 45, n_samples)
seche_duty  = rng_sim.uniform(0.55, 0.65, n_samples)

# 3. Recharge VE : ~3700 W, très long, variance quasi nulle (charge continue)
ve_p_max = rng_sim.normal(3700, 40, n_samples)
ve_duree = rng_sim.uniform(180, 360, n_samples)
ve_std   = rng_sim.normal(45, 10, n_samples)
ve_duty  = np.full(n_samples, 1.0)

df_appareils = pd.DataFrame({
    'puissance_max_W': np.concatenate([cuit_p_max, seche_p_max, ve_p_max]),
    'ecart_type_W': np.concatenate([cuit_std, seche_std, ve_std]),
    'duree_min': np.concatenate([cuit_duree, seche_duree, ve_duree]),
    'duty_cycle': np.concatenate([cuit_duty, seche_duty, ve_duty]),
    'appareil': ['cuit_vapeur']*n_samples + ['seche_linge']*n_samples + ['recharge_ve']*n_samples
})

print(f"Jeu de données synthétique généré : {df_appareils.shape[0]} sessions réparties en 3 classes :")
display(df_appareils.groupby('appareil').mean().round(1))


In [ ]:
# Programme 8 — Exercice 2
# À toi de jouer : sépare les features X et la cible y, puis fais le train_test_split :

X = ...
y = ...

X_train, X_test, y_train, y_test = train_test_split(..., test_size=0.3, random_state=42, stratify=y)


In [ ]:
# Programme 9 — Test Exercice 2
# Exécutez cette cellule pour valider vos réponses
check("X_train shape", X_train.shape, (105, 4))
check("X_test shape", X_test.shape, (45, 4))
check("Classes dans y", sorted(list(y.unique())), ['cuit_vapeur', 'recharge_ve', 'seche_linge'])


---
## 4. Entraînement et Visualisation de l'Arbre avec Scikit-Learn (`plot_tree`)

### 🌲 Instancier et entraîner un DecisionTreeClassifier
Dans `scikit-learn`, un arbre de décision se crée en quelques lignes :
```python
clf = DecisionTreeClassifier(criterion='gini', max_depth=2, random_state=42)
clf.fit(X_train, y_train)
```

### 🖼️ Visualiser l'arbre complet avec `plot_tree`
Pas besoin d'installer des outils externes complexes comme Graphviz ! Scikit-Learn intègre la fonction **`plot_tree`** qui dessine l'arbre directement avec Matplotlib :
```python
fig, ax = plt.subplots(figsize=(10, 6))
plot_tree(clf, feature_names=X.columns, class_names=clf.classes_, filled=True, rounded=True)
plt.show()
```

Chaque boîte affiche :
- La condition de split (ex: `puissance_max_W <= 1774.5`)
- L'indice `gini` actuel du nœud
- Le nombre total d'échantillons `samples` présents dans le nœud
- La répartition des classes `value = [cuit, ve, seche]`
- La classe majoritaire assignée `class`
- La couleur : plus la teinte est vive, plus le nœud est pur !

---

### ✏️ Exercice 3 — Entraînement de l'arbre de décision

1. Créez un arbre `modele_arbre = DecisionTreeClassifier(criterion='gini', max_depth=2, random_state=42)`.
2. Entraînez-le sur le jeu d'entraînement avec `.fit(X_train, y_train)`.
3. Récupérez le nom de la feature ayant la plus grande importance dans la décision (`modele_arbre.feature_importances_`) et stockez-le dans `feature_racine`.

<details>
<summary>💡 Cliquez ici pour afficher un indice ou la solution</summary>

```python
modele_arbre = DecisionTreeClassifier(criterion='gini', max_depth=2, random_state=42)
modele_arbre.fit(X_train, y_train)

importances = pd.Series(modele_arbre.feature_importances_, index=X.columns)
feature_racine = importances.idxmax()
```
</details>


In [ ]:
# Programme 10 — Démonstration : Entraînement et visualisation de l'arbre
demo_clf = DecisionTreeClassifier(criterion='gini', max_depth=2, random_state=42)
demo_clf.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(12, 6), dpi=120)
plot_tree(demo_clf, 
          feature_names=list(X.columns), 
          class_names=list(demo_clf.classes_), 
          filled=True, 
          rounded=True, 
          fontsize=10, 
          ax=ax)
plt.title("Arbre de Décision Entraîné (Profondeur = 2)", fontweight='bold', pad=15)
plt.show()


In [ ]:
# Programme 11 — Exercice 3
# À toi de jouer : instancie, entraîne l'arbre et identifie la feature la plus importante :

modele_arbre = DecisionTreeClassifier(...)
modele_arbre.fit(...)

feature_racine = ...


In [ ]:
# Programme 12 — Test Exercice 3
# Exécutez cette cellule pour valider vos réponses
check("Profondeur de l'arbre", modele_arbre.get_depth(), 2)
check("Feature racine", feature_racine, "puissance_max_W")
check("Classes du modèle", list(modele_arbre.classes_), ['cuit_vapeur', 'recharge_ve', 'seche_linge'])


---
## 5. Évaluation des Performances et Matrice de Confusion

### 🎯 La Matrice de Confusion : Où le modèle se trompe-t-il ?
L'**Accuracy** (taux de bonnes prédictions $\frac{\text{Bien classés}}{\text{Total}}$) ne suffit pas : elle ne dit pas si le modèle confond le sèche-linge avec la voiture ou avec le cuit-vapeur.
La **Matrice de Confusion** croise :
- En **lignes** : Les vraies classes réelles du jeu de test.
- En **colonnes** : Les prédictions faites par le modèle.
- La **diagonale principale** représente les succès ; tous les éléments hors-diagonale sont des erreurs de confusion !

---

### 🥊 Comparer avec des modèles de référence (Baselines)
Un score d'exactitude de 75% est-il bon ? Pour en juger, on compare toujours avec deux baselines :
1. **Le `DummyClassifier`** : Modèle trivial qui prédit aveuglément la classe la plus fréquente. Avec 3 classes équilibrées, son score est de $33.3\%$ (pur hasard).
2. **Le `KNeighborsClassifier` (KNN)** : Classifieur géométrique basé sur les $k$ plus proches voisins (présenté dans le Tutorial 5).

---

### ✏️ Exercice 4 — Évaluation et comparaison des modèles

1. Calculez l'accuracy de `modele_arbre` sur le jeu de test avec `.score(X_test, y_test)` et stockez-la dans `acc_arbre`.
2. Instanciez un `DummyClassifier(strategy='most_frequent')`, entraînez-le sur `(X_train, y_train)` et calculez son score sur `(X_test, y_test)` dans `acc_dummy`.
3. Instanciez un `KNeighborsClassifier(n_neighbors=3)`, entraînez-le sur `(X_train, y_train)` et calculez son score sur `(X_test, y_test)` dans `acc_knn`.

<details>
<summary>💡 Cliquez ici pour afficher un indice ou la solution</summary>

```python
acc_arbre = float(modele_arbre.score(X_test, y_test))

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
acc_dummy = float(dummy.score(X_test, y_test))

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)
acc_knn = float(knn.score(X_test, y_test))
```
</details>


In [ ]:
# Programme 13 — QCM 2
qcm(
    "Dans une matrice de confusion 3x3 pour la détection d'appareils, que représente un chiffre situé en dehors de la diagonale principale ?",
    [
        "Une observation où le modèle a prédit correctement la classe.",
        "Une erreur de prédiction (le modèle a confondu la classe réelle de la ligne avec la classe prédite de la colonne).",
        "Le nombre de valeurs manquantes (NaN) dans le jeu de test.",
        "Le nombre d'arbres nécessaires dans la forêt aléatoire."
    ],
    correct=1,
    explication="Exactement ! La diagonale principale comptabilise les succès (vrais positifs). Toute valeur hors-diagonale (ligne i, colonne j avec i != j) représente une erreur de classification où un appareil i a été pris à tort pour un appareil j."
)


In [ ]:
# Programme 14 — Démonstration : Matrice de confusion affichée
y_pred_demo = demo_clf.predict(X_test)
cm = confusion_matrix(y_test, y_pred_demo, labels=demo_clf.classes_)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=demo_clf.classes_)
disp.plot(cmap='Blues', ax=ax, values_format='d')
plt.title("Matrice de Confusion sur le Jeu de Test (45 sessions)", fontweight='bold', pad=12)
plt.grid(False)
plt.show()


In [ ]:
# Programme 15 — Exercice 4
# À toi de jouer : calcule les scores de l'arbre, du Dummy et du KNN :

acc_arbre = ...

dummy = DummyClassifier(...)
dummy.fit(...)
acc_dummy = ...

knn = KNeighborsClassifier(...)
knn.fit(...)
acc_knn = ...


In [ ]:
# Programme 16 — Test Exercice 4
# Exécutez cette cellule pour valider vos réponses
check("acc_arbre", acc_arbre, 1.0, tol=1e-3)
check("acc_dummy", round(acc_dummy, 3), 0.333, tol=0.01)
check("acc_knn", acc_knn, 1.0, tol=1e-3)


---
## 6. Le Risque de Surapprentissage (*Overfitting*) et les Hyperparamètres

### 🐍 Le piège mortel de l'arbre sans contrainte
Si l'on ne fixe aucune limite de profondeur (`max_depth=None`), un arbre de décision continuera de diviser les nœuds jusqu'à ce que **chaque feuille ne contienne qu'un seul exemple**.
- Il obtiendra un score parfait de **100% sur le train set** en apprenant par cœur chaque point bruité.
- Mais face à de nouvelles données inconnues (test set), il s'effondrera : c'est le **surapprentissage (*overfitting*)**.

### 🎛️ Les 3 hyperparamètres clés de régularisation (Élagage / Pruning) :
1. **`max_depth`** : Profondeur maximale de l'arbre (le nombre maximal de questions en cascade). C'est le levier n°1 contre l'overfitting.
2. **`min_samples_split`** : Nombre minimal d'échantillons requis dans un nœud pour autoriser une nouvelle division (défaut = 2). Augmenter ce seuil (ex: 10) empêche les divisions trop opportunistes.
3. **`min_samples_leaf`** : Nombre minimal d'échantillons devant obligatoirement résider dans chaque feuille terminale (défaut = 1).


In [ ]:
# Programme 17 — Démonstration : Courbe d'apprentissage Train vs Test selon la profondeur
profondeurs = range(1, 8)
scores_train = []
scores_test = []

for d in profondeurs:
    clf_d = DecisionTreeClassifier(max_depth=d, random_state=42)
    clf_d.fit(X_train, y_train)
    scores_train.append(clf_d.score(X_train, y_train))
    scores_test.append(clf_d.score(X_test, y_test))

plt.figure(figsize=(9, 4.5))
plt.plot(profondeurs, scores_train, marker='o', linewidth=2, color='#1d3557', label="Score Train Set")
plt.plot(profondeurs, scores_test, marker='s', linewidth=2, linestyle='--', color='#e63946', label="Score Test Set")
plt.axvline(2, color='green', linestyle=':', linewidth=2, label="Profondeur optimale (d=2)")
plt.title("Évolution de l'Accuracy en fonction de la profondeur de l'arbre", fontweight='bold')
plt.xlabel("Profondeur maximale (max_depth)")
plt.ylabel("Exactitude (Accuracy)")
plt.ylim(0.5, 1.05)
plt.legend()
plt.tight_layout()
plt.show()


---
## 7. 🏆 Défi Final : La Classification d'une Session Mystère

Un auditeur énergétique relève une session anonyme sur le compteur Linky d'un logement :
- **Puissance maximale** : $3\,680\text{ W}$
- **Écart-type de la puissance** : $50\text{ W}$
- **Durée de la session** : $210\text{ minutes}$ (3h30)
- **Duty cycle** : $1.0$ (signal 100% continu)

À l'aide de votre modèle `modele_arbre` entraîné précédemment, résolvez les 4 questions d'audit :

1. **`classe_mystere`** : Quelle est la classe prédite par l'arbre pour cette session ? (`str`).
2. **`feature_la_plus_importante`** : Quelle est la variable ayant reçu la plus grande importance dans l'arbre ? (`str`).
3. **`profondeur_optimale`** : Quelle est la profondeur minimale (`int`) requise pour séparer parfaitement nos 3 classes d'appareils ?
4. **`accuracy_finale_test`** : Quel est le taux d'exactitude (Accuracy entre 0.0 et 1.0) obtenu par l'arbre sur le jeu de test ?

<details>
<summary>💡 Cliquez ici pour afficher un indice ou la solution</summary>

```python
session_mystere = pd.DataFrame([{
    'puissance_max_W': 3680.0,
    'ecart_type_W': 50.0,
    'duree_min': 210.0,
    'duty_cycle': 1.0
}])

classe_mystere = str(modele_arbre.predict(session_mystere)[0])

importances = pd.Series(modele_arbre.feature_importances_, index=X.columns)
feature_la_plus_importante = str(importances.idxmax())

profondeur_optimale = 2
accuracy_finale_test = float(modele_arbre.score(X_test, y_test))
```
</details>


In [ ]:
# Programme 18 — Défi Final
# À toi de jouer : réponds aux 4 questions du rapport d'audit :

classe_mystere = ...
feature_la_plus_importante = ...
profondeur_optimale = ...
accuracy_finale_test = ...


In [ ]:
# Programme 19 — Test Défi Final
# Exécutez cette cellule pour valider l'ensemble de votre audit décisionnel !
check("1. Classe mystère prédite", classe_mystere, "recharge_ve")
check("2. Feature la plus importante", feature_la_plus_importante, "puissance_max_W")
check("3. Profondeur minimale optimale", profondeur_optimale, 2)
check("4. Accuracy finale sur test", accuracy_finale_test, 1.0, tol=1e-3)


---
## 🎓 Synthèse de la Section 9 : Ce que vous maîtrisez désormais

Félicitations ! Vous venez d'ajouter un pilier central du Machine Learning à votre boîte à outils :

1. **La géométrie des arbres** : Vous comprenez pourquoi les découpages orthogonaux posent des questions claires sur une variable à la fois, garantissant une **explicabilité totale**.
2. **La sélection mathématique des splits** : Vous maîtrisez le calcul de l'**Impureté de Gini** et comprenez comment l'algorithme trouve automatiquement les seuils optimaux.
3. **L'évaluation rigoureuse** : Vous savez lire une **Matrice de Confusion** et comparer vos modèles à des baselines standards (`DummyClassifier`, `KNN`).
4. **La maîtrise du surapprentissage** : Vous savez réguler la complexité d'un arbre grâce à `max_depth` et aux contraintes de division.

---

### 🚀 Pourquoi avons-nous besoin de la Section 10 (Random Forest) ?
Un arbre de décision unique est élégant et rapide, mais il souffre d'un défaut structurel : **la forte variance**. Une petite modification de quelques points dans le jeu d'entraînement peut faire basculer la première question posée et changer complètement la structure de l'arbre.

Dans la **Section 10 (La Maison Complète)**, nous allons lever cette limite en découvrant le **Random Forest** : au lieu d'un seul arbre, nous entraînerons une forêt de centaines d'arbres indépendants dont le vote collectif élimine le bruit pour classifier l'ensemble des appareils de la maison !
